In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys

PROJECT_ROOT = "/content/drive/MyDrive/MultilingualFakeNews"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project ready.")

Mounted at /content/drive
Project ready.


In [2]:
import torch
from transformers import AutoTokenizer

from src.model import XLMRClassifier
from src.preprocessing import clean_text
from src.config import *

In [3]:
import importlib
import src.inference

importlib.reload(src.inference)

print("Inference module imported successfully.")

Inference module imported successfully.


In [4]:
import importlib
import src.inference

importlib.reload(src.inference)

print("Inference module imports successfully.")

Inference module imports successfully.


In [5]:
#LAZY LOADING

"""
Inference module for the Multilingual Fake News Detection system.
"""

import torch
from transformers import AutoTokenizer

from src.model import XLMRClassifier
from src.preprocessing import clean_text
from src.config import *


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = None
tokenizer = None


def load_model():
    """
    Load the trained model and tokenizer once.
    """

    global model
    global tokenizer

    if model is None:

        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )

        model = XLMRClassifier()

        model.load_state_dict(
            torch.load(
                f"{MODEL_DIR}/best_xlmr.pt",
                map_location=device
            )
        )

        model.to(device)

        model.eval()

    return model, tokenizer

In [ ]:
with open(
    "/content/drive/MyDrive/MultilingualFakeNews/src/inference.py",
    "r"
) as f:
    print(f.read())

"""
Inference module for the Multilingual Fake News Detection system.
"""

import torch
from transformers import AutoTokenizer

from src.model import XLMRClassifier
from src.preprocessing import clean_text
from src.config import *


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Cached objects
_model = None
_tokenizer = None


def load_model():
    """
    Load the tokenizer and trained model once.
    """

    global _model
    global _tokenizer

    if _model is None:

        print("Loading tokenizer...")

        _tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )

        print("Loading trained model...")

        _model = XLMRClassifier()

        checkpoint = torch.load(
            f"{MODEL_DIR}/best_xlmr.pt",
            map_location=device
        )

        _model.load_state_dict(checkpoint)

        _model.to(device)

        _model.eval()

        print("Model ready.")

    return _model, _tokenizer
    
import torch

In [7]:
import importlib
import src.inference

importlib.reload(src.inference)

model, tokenizer = src.inference.load_model()

print(type(model))
print(type(tokenizer))

Loading tokenizer...
Loading trained model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.
<class 'src.model.XLMRClassifier'>
<class 'transformers.models.xlm_roberta.tokenization_xlm_roberta.XLMRobertaTokenizer'>


In [8]:
# Reloading the model
import importlib
import src.inference

importlib.reload(src.inference)

<module 'src.inference' from '/content/drive/MyDrive/MultilingualFakeNews/src/inference.py'>

In [14]:
# Testing the model
news = """
NASA has confirmed the discovery of liquid water beneath the surface of Mars.
Scientists believe the discovery could increase the chances of finding microbial life.
"""

result = src.inference.predict_news(news)

print(result)

Loading tokenizer...
Loading trained model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.
{'prediction': 'Legit', 'label_id': 0, 'confidence': 0.9819, 'is_confident': True, 'probabilities': {'Legit': 0.9819, 'Fake': 0.0181}, 'model': 'xlm-roberta-base', 'max_length': 384}


In [10]:
# Testing the model, 2
news = """
Aliens have secretly taken over the United Nations and are controlling every government on Earth.
"""

result = src.inference.predict_news(news)

print(result)

Loading tokenizer...
Loading trained model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.
{'prediction': 'Fake', 'label_id': 1, 'confidence': 0.8597, 'is_confident': True, 'probabilities': {'Legit': 0.1403, 'Fake': 0.8597}, 'model': 'xlm-roberta-base', 'max_length': 384, 'model_version': '1.0.0'}


In [15]:
from src.inference import predict_news

result = predict_news(
    "NASA discovers water beneath Mars."
)

print(result)

{'prediction': 'Legit', 'label_id': 0, 'confidence': 0.9671, 'is_confident': True, 'probabilities': {'Legit': 0.9671, 'Fake': 0.0329}, 'model': 'xlm-roberta-base', 'max_length': 384}


In [9]:
from src.inference import predict_dataframe

In [19]:
# Testing batch prediction

import pandas as pd

sample_df = pd.DataFrame({
    "text": [
        "NASA discovers water beneath Mars.",
        "Aliens secretly control the United Nations.",
        "Roger Federer wins another tennis tournament."
    ]
})

results = predict_dataframe(sample_df)

results

,text,Prediction,Confidence
0,NASA discovers water beneath Mars.,Legit,0.9671
1,Aliens secretly control the United Nations.,Legit,0.5920
2,Roger Federer wins another tennis tournament.,Legit,0.9912
